# Level 1 — log-Mel CNN

Training and evaluation in one place, using the same code as the scripts
(`src.training.train`, `src.evaluation.*`), so the notebook and the CLI can
never drift apart.

The question this notebook answers is **not** "how accurate is the detector".
It is: *how much worse does it get on generators it never trained on?* Training
uses G01–G04 only; G05–G07 exist solely in the test split.

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = "https://github.com/Neha-Jacob-8/environmental-sound-deepfake-detection.git"
    ROOT = Path("/content/speech")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch", "torchaudio", "soundfile", "pandas", "numpy",
                    "matplotlib", "scikit-learn"], check=True)
else:
    ROOT = Path.cwd()
    if not (ROOT / "src").exists():
        ROOT = ROOT.parent

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt

from src.datasets.envsdd_dataset import make_loader, logmel_stats
from src.models.cnn import LogMelCNN
from src.training.train import pick_device, set_seed, train_one_epoch, score_loader
from src.evaluation.metrics import eer, all_metrics, per_generator_eer, seen_unseen_summary
from src.preprocessing.generators import (
    SEEN_GENERATORS, UNSEEN_GENERATORS, GENERATOR_NAMES)

DEVICE = pick_device()
print("root  :", ROOT)
print("device:", DEVICE)

# --- chart styling ------------------------------------------------------
# Three categorical slots, validated for colour-vision deficiency at all pairs.
C_REAL, C_SEEN, C_UNSEEN = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#dedddb"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "font.size": 10,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110,
})

## 2. What the model sees

Standardisation stats come from the **train** split only and are cached, so
validation and test statistics never leak into training. A standardised feature
should sit near mean 0 / std 1 on train; the test split lands slightly off,
which is correct — it has source domains (Clotho, DCASE2023Task7) that train
never sees, and re-standardising per split would erase exactly the shift under
study.

In [ ]:
mean, std = logmel_stats()
print(f"per-mel-bin mean  {mean.min():7.2f} .. {mean.max():7.2f}")
print(f"per-mel-bin std   {std.min():7.2f} .. {std.max():7.2f}\n")

loaders = {
    "train":      make_loader("train", mode="logmel", batch_size=32, num_workers=0),
    "validation": make_loader("validation", mode="logmel", batch_size=64,
                              shuffle=False, num_workers=0),
    "test":       make_loader("test", mode="logmel", batch_size=64,
                              shuffle=False, num_workers=0),
}
for name, ld in loaders.items():
    print(ld.dataset.describe())

xb, yb = next(iter(loaders["train"]))
print(f"\nbatch {tuple(xb.shape)}  mean {xb.mean():+.3f}  std {xb.std():.3f}")

## 3. The model

Four conv blocks → global average pooling → one logit. Global pooling rather
than a flatten: a flatten would tie the classifier to a fixed 251-frame input
and let it learn *where* in the clip an artifact sits. Artifacts are a property
of the whole clip.

Kept small on purpose. With 1,200 training source recordings, a bigger network
would memorise the recordings rather than the generator artifacts — and that is
precisely the failure that collapses on unseen generators.

In [ ]:
model = LogMelCNN().to(DEVICE)
print(model)
print(f"\ntrainable parameters: {model.n_params():,}")
print(f"feature map before pooling: {tuple(model.features(xb.to(DEVICE)).shape)}")
print(f"embedding (Level 3 fusion will reuse this): {tuple(model.embed(xb.to(DEVICE)).shape)}")

## 4. Training

Model selection is on **validation EER**, not validation loss — loss is
dominated by the easy majority of clips, and the two do not peak at the same
epoch.

One caveat that shapes how everything below must be read: validation contains
only G01–G04, the same generators as training. It can tell you when the model
has finished learning the *seen* generators. It cannot be used to select for
generalisation — that would tune on the very thing the test split measures.

In [ ]:
EPOCHS, PATIENCE, LR, SEED = 40, 8, 1e-3, 1337
CKPT = "results/models/cnn_best.pt"

# Set False to skip training and load the existing checkpoint instead.
TRAIN_NOW = True

In [ ]:
import time
import torch.nn as nn

set_seed(SEED)
model = LogMelCNN().to(DEVICE)
pos_weight = loaders["train"].dataset.class_weights().to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)

Path("results/models").mkdir(parents=True, exist_ok=True)
Path("results/tables").mkdir(parents=True, exist_ok=True)

if TRAIN_NOW:
    print(f"device={DEVICE}  params={model.n_params():,}  pos_weight={pos_weight.item():.3f}")
    print(f"{'ep':>3} {'train_loss':>11} {'val_loss':>9} {'val_eer':>8} {'val_auc':>8} {'sec':>6}")

    best_eer, best_epoch, history = float("inf"), -1, []
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, loaders["train"], opt, criterion, DEVICE)
        s, y, val_loss = score_loader(model, loaders["validation"], DEVICE, criterion)
        m = all_metrics(y, s)
        sched.step(val_loss)
        dt = time.time() - t0

        print(f"{ep:3d} {tr_loss:11.4f} {val_loss:9.4f} {m['eer']:8.4f} "
              f"{m['auc']:8.4f} {dt:6.1f}" + ("  *" if m['eer'] < best_eer else ""))
        history.append(dict(epoch=ep, train_loss=tr_loss, val_loss=val_loss,
                            val_eer=m["eer"], val_auc=m["auc"], seconds=dt))

        if m["eer"] < best_eer:
            best_eer, best_epoch = m["eer"], ep
            torch.save({"state_dict": model.state_dict(), "model": "LogMelCNN",
                        "model_kwargs": {}, "epoch": ep, "val_eer": best_eer,
                        "args": {"epochs": EPOCHS, "lr": LR, "seed": SEED}}, CKPT)
        elif ep - best_epoch >= PATIENCE:
            print(f"\nearly stop: no val-EER improvement in {PATIENCE} epochs")
            break

    hist = pd.DataFrame(history)
    hist.to_csv("results/tables/cnn_history.csv", index=False)
    print(f"\nbest val EER {best_eer:.4f} at epoch {best_epoch}")
else:
    hist = pd.read_csv("results/tables/cnn_history.csv")
    print("skipped training, loaded existing history")

ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ck["state_dict"]); model.to(DEVICE).eval()
print(f"loaded checkpoint: epoch {ck['epoch']}, val EER {ck['val_eer']:.4f}")

## 5. Training curves

Two panels rather than two y-axes on one plot: loss and EER are different
quantities on different scales, and overlaying them on twin axes makes the
crossing point look meaningful when it is an artifact of the scaling.

In [ ]:
best_ep = int(hist.loc[hist.val_eer.idxmin(), "epoch"])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True,
                               gridspec_kw={"height_ratios": [1, 1], "hspace": 0.12})

ax1.plot(hist.epoch, hist.train_loss, lw=2, color=C_SEEN, label="train loss")
ax1.plot(hist.epoch, hist.val_loss, lw=2, color=C_REAL, label="validation loss")
ax1.set_ylabel("BCE loss"); ax1.set_yscale("log")
ax1.legend(frameon=False, loc="upper right")

ax2.plot(hist.epoch, hist.val_eer, lw=2, color=C_REAL)
ax2.scatter([best_ep], [hist.val_eer.min()], s=60, zorder=5,
            color=C_REAL, edgecolor="#fcfcfb", linewidth=2)
ax2.annotate(f"best {hist.val_eer.min():.4f}  (epoch {best_ep})",
             (best_ep, hist.val_eer.min()), textcoords="offset points",
             xytext=(10, 10), color=INK, fontsize=9)
ax2.set_ylabel("validation EER"); ax2.set_xlabel("epoch")

for ax in (ax1, ax2):
    ax.axvline(best_ep, color=GRID, lw=1, zorder=0)
fig.suptitle("Level 1 CNN — training curves", x=0.09, ha="left", fontsize=13)
plt.show()

hist[["epoch", "train_loss", "val_loss", "val_eer", "val_auc"]].tail(8)

## 6. The result — test split

Every generator is scored against the **same** 300 real clips, so the only thing
that changes between rows is the fake side. Both the seen and unseen pools come
from the test split too, so they share a source-domain mix — without that, an
apparent generalisation failure could just be the test split's unfamiliar source
domains showing up.

In [ ]:
scores, labels, _ = score_loader(model, loaders["test"], DEVICE)
gens = loaders["test"].dataset.df.generator.to_numpy()
assert len(gens) == len(scores)

per_gen = per_generator_eer(labels, scores, gens)
summary = seen_unseen_summary(labels, scores, gens, SEEN_GENERATORS, UNSEEN_GENERATORS)

tbl = pd.DataFrame([
    {"generator": g, "name": GENERATOR_NAMES[g],
     "seen": "seen" if g in SEEN_GENERATORS else "UNSEEN",
     "EER": m["eer"], "AUC": m["auc"], "F1": m["f1"]}
    for g, m in sorted(per_gen.items())
]).set_index("generator")

print(f"pooled seen   (G01-G04)  EER {summary['seen']['eer']:.4f}")
print(f"pooled unseen (G05-G07)  EER {summary['unseen']['eer']:.4f}")
print(f"generalisation gap       {summary['gap']:+.4f}")
tbl.round(4)

In [ ]:
order = sorted(per_gen)
vals = [per_gen[g]["eer"] for g in order]
cols = [C_SEEN if g in SEEN_GENERATORS else C_UNSEEN for g in order]

fig, ax = plt.subplots(figsize=(9, 4.2))
bars = ax.bar(order, vals, color=cols, width=0.62, zorder=3)
for b, v in zip(bars, vals):                      # direct labels, always
    ax.text(b.get_x() + b.get_width() / 2, v + 0.0025, f"{v:.4f}",
            ha="center", fontsize=9, color=INK)

# Pooled levels carry their value in the legend rather than as floating text,
# which would collide with whichever bar happens to sit at that height.
l_seen = ax.axhline(summary["seen"]["eer"], color=C_SEEN, lw=1.5, ls="--", zorder=2)
l_unseen = ax.axhline(summary["unseen"]["eer"], color=C_UNSEEN, lw=1.5, ls="--", zorder=2)

ax.set_ylabel("EER  (lower is better)")
ax.set_ylim(0, max(vals) * 1.32)
ax.set_xlim(-0.6, 6.6)
ax.set_axisbelow(True); ax.xaxis.grid(False)

handles = [plt.Rectangle((0, 0), 1, 1, color=C_SEEN),
           plt.Rectangle((0, 0), 1, 1, color=C_UNSEEN), l_seen, l_unseen]
# Named legend_labels, not labels: `labels` holds the ground-truth array that
# the later cells index `scores` with, and shadowing it breaks them silently.
legend_labels = ["seen at training (G01-G04)", "UNSEEN (G05-G07)",
                 f"pooled seen  {summary['seen']['eer']:.4f}",
                 f"pooled unseen  {summary['unseen']['eer']:.4f}"]
ax.legend(handles, legend_labels, frameon=False, loc="upper left",
          ncols=2, fontsize=9)
ax.set_title("Per-generator EER on the test split",
             loc="left", fontsize=12, color=INK, pad=12)
plt.show()

## 7. Where it actually fails

Three views of the same scores. The histogram is the most informative: it shows
*why* the EER rises, not just that it does.

In [ ]:
real_s = scores[labels == 0]
seen_s = scores[np.isin(gens, SEEN_GENERATORS)]
unseen_s = scores[np.isin(gens, UNSEEN_GENERATORS)]

fig, ax = plt.subplots(figsize=(9.5, 4))
bins = np.linspace(min(scores), max(scores), 70)
for arr, c, lab in ((real_s, C_REAL, "real"),
                    (seen_s, C_SEEN, "fake — seen G01-G04"),
                    (unseen_s, C_UNSEEN, "fake — UNSEEN G05-G07")):
    ax.hist(arr, bins=bins, color=c, alpha=0.62, label=lab, zorder=3)

thr = eer(labels, scores)[1]
ax.axvline(thr, color=INK, lw=1.5, ls="--", zorder=4)
ax.text(thr, ax.get_ylim()[1] * 0.95, f"  EER threshold {thr:.2f}",
        fontsize=9, color=INK, va="top")
ax.set_xlabel("model output (logit) — higher means 'fake'")
ax.set_ylabel("clips"); ax.set_axisbelow(True); ax.xaxis.grid(False)
ax.legend(frameon=False, loc="upper left")
ax.set_title("Unseen fakes drift left, toward the real distribution",
             loc="left", fontsize=12, pad=12)
plt.show()

print(f"median logit   real {np.median(real_s):+7.2f}   "
      f"seen fake {np.median(seen_s):+7.2f}   unseen fake {np.median(unseen_s):+7.2f}")

In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix

fig, (axA, axB) = plt.subplots(1, 2, figsize=(12.5, 4.8),
                               gridspec_kw={"wspace": 0.25})

# --- ROC: pooled seen vs pooled unseen ---------------------------------
real_m = labels == 0
for group, c, lab in ((SEEN_GENERATORS, C_SEEN, "seen (G01-G04)"),
                      (UNSEEN_GENERATORS, C_UNSEEN, "UNSEEN (G05-G07)")):
    sel = real_m | np.isin(gens, group)
    fpr, tpr, _ = roc_curve(labels[sel], scores[sel])
    e = eer(labels[sel], scores[sel])[0]
    axA.plot(fpr, tpr, lw=2, color=c, label=f"{lab}   EER {e:.4f}")
axA.plot([0, 1], [1, 0], color=GRID, lw=1, ls=":")     # the EER line
axA.text(0.62, 0.42, "EER line", color=INK2, fontsize=8.5, rotation=-45)
axA.set_xlabel("false positive rate"); axA.set_ylabel("true positive rate")
axA.legend(frameon=False, loc="lower right")
axA.set_title("ROC", loc="left", fontsize=12, pad=10)

# --- confusion matrix at the validation-chosen threshold ---------------
val_s, val_y, _ = score_loader(model, loaders["validation"], DEVICE)
thr_val = eer(val_y, val_s)[1]
cm = confusion_matrix(labels, (scores >= thr_val).astype(int))

from matplotlib.colors import LinearSegmentedColormap
ramp = LinearSegmentedColormap.from_list("blues", ["#fcfcfb", C_REAL])
axB.imshow(cm / cm.sum(axis=1, keepdims=True), cmap=ramp, vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        frac = cm[i, j] / cm[i].sum()
        axB.text(j, i, f"{cm[i, j]}\n{frac:.1%}", ha="center", va="center",
                 fontsize=12, color="#fcfcfb" if frac > 0.5 else INK)
axB.set_xticks([0, 1], ["pred real", "pred fake"])
axB.set_yticks([0, 1], ["true real", "true fake"])
axB.grid(False)
axB.set_title(f"Confusion matrix  (threshold {thr_val:.3f}, chosen on validation)",
              loc="left", fontsize=12, pad=10)
plt.show()

## 8. Is the gap real?

At 300 test source groups the error bars are wide enough to matter, so the gap
needs a confidence interval before it can be claimed.

The bootstrap resamples **whole source groups**, not individual clips. Clips
inside a group come from one recording and their scores are correlated;
resampling clips would treat them as independent and report intervals that are
too narrow.

In [ ]:
sid = loaders["test"].dataset.df.source_id.to_numpy()
groups = np.unique(sid)
by_group = {g: np.flatnonzero(sid == g) for g in groups}
rng = np.random.default_rng(0)

seen_m = real_m | np.isin(gens, SEEN_GENERATORS)
unseen_m = real_m | np.isin(gens, UNSEEN_GENERATORS)

def pooled(mask, idx):
    sel = idx[mask[idx]]
    return eer(labels[sel], scores[sel])[0]

N_BOOT = 2000
boot = np.empty((N_BOOT, 2))
for b in range(N_BOOT):
    idx = np.concatenate([by_group[g] for g in rng.choice(groups, len(groups), replace=True)])
    boot[b] = pooled(seen_m, idx), pooled(unseen_m, idx)
gap = boot[:, 1] - boot[:, 0]

for name, point, col in (("seen   (G01-G04)", summary["seen"]["eer"], boot[:, 0]),
                         ("unseen (G05-G07)", summary["unseen"]["eer"], boot[:, 1])):
    lo, hi = np.percentile(col, [2.5, 97.5])
    print(f"{name:18s} EER {point:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")
lo, hi = np.percentile(gap, [2.5, 97.5])
print(f"{'GAP':18s}     {summary['gap']:+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]")
print(f"\nP(gap > 0) = {(gap > 0).mean():.4f}   over {N_BOOT} resamples")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.hist(gap, bins=50, color=C_UNSEEN, alpha=0.75, zorder=3)
ax.axvline(0, color=INK, lw=1.5, zorder=4)
ax.axvline(summary["gap"], color=C_SEEN, lw=2, zorder=5)
ax.text(summary["gap"], ax.get_ylim()[1] * 0.92,
        f"  observed {summary['gap']:+.4f}", color=C_SEEN, fontsize=9, va="top")
lo, hi = np.percentile(gap, [2.5, 97.5])
ax.axvspan(lo, hi, color=C_UNSEEN, alpha=0.12, zorder=1)
ax.set_xlabel("unseen EER − seen EER"); ax.set_ylabel("bootstrap resamples")
ax.set_axisbelow(True); ax.xaxis.grid(False)
ax.set_title(f"Generalisation gap — 95% CI [{lo:+.4f}, {hi:+.4f}], never crosses zero",
             loc="left", fontsize=12, pad=10)
plt.show()

## Summary

| | EER |
|---|---|
| validation (G01–G04) | best from training above |
| test, seen G01–G04 | pooled seen |
| test, **unseen G05–G07** | pooled unseen |

Three things worth carrying into Level 2:

1. **It is not classic overfitting.** Train and validation accuracy sit within
   0.03pp of each other. More dropout or more regularisation has no gap to
   close — the failure is across *generators*, an axis validation cannot see.
2. **Recall is what breaks.** Precision holds on the test split while recall
   falls on the unseen generators: roughly one unseen fake in seven is passed
   as real. That is the expensive direction for a detector.
3. **All three unseen generators land within 0.007 of each other.** The
   detector does not fail on particular architectures — it degrades the moment
   the generator is out of distribution. That is a cleaner motivation for the
   fusion work than a per-generator scatter would have been.

Next: Level 2, AASIST on raw waveforms — `EnvSDDDataset(mode="waveform")`
already serves `(1, 64000)`.